In [4]:
import time
import requests
import networkx as nx
import pandas as pd
import os


In [ ]:

SEED_ARTICLE = "Arsenal F.C."
MAX_LINKS    = 80
REQUEST_DELAY = 0.5

WIKI_API       = "https://en.wikipedia.org/api/rest_v1"
WIKI_QUERY_API = "https://en.wikipedia.org/w/api.php"


HEADERS = {
    "User-Agent": "WikiNetworkAnalysis/1.0 (academic research; anonymous)"
}


In [6]:
def get_outgoing_links(title: str, max_links: int) -> list[str]:
    """
    Fetch all Wikipedia articles that a given article links TO.
    Uses the MediaWiki query API with the 'links' property.
    """
    links = []
    params = {
        "action":      "query",
        "titles":      title,
        "prop":        "links",
        "pllimit":     "max",
        "plnamespace": 0,
        "format":      "json",
    }

    print(f"  Fetching links from '{title}'...")

    while len(links) < max_links:
        response = requests.get(
            WIKI_QUERY_API, params=params, headers=HEADERS, timeout=10
        )
        response.raise_for_status()
        data = response.json()

        pages = data.get("query", {}).get("pages", {})
        for page in pages.values():
            for link in page.get("links", []):
                links.append(link["title"])
                if len(links) >= max_links:
                    return links

        if "continue" not in data:
            break
        params["plcontinue"] = data["continue"]["plcontinue"]
        time.sleep(REQUEST_DELAY)

    return links[:max_links]


def get_article_summary(title: str) -> dict:
    """
    Fetch a short summary and metadata for a Wikipedia article.
    Used to enrich node attributes in the graph.
    """
    try:
        url = f"{WIKI_API}/page/summary/{requests.utils.quote(title)}"
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return {
                "title":       data.get("title", title),
                "description": data.get("description", ""),
                "url":         data.get("content_urls", {})
                                   .get("desktop", {})
                                   .get("page", ""),
                "extract":     data.get("extract", "")[:200],
            }
    except Exception:
        pass
    return {"title": title, "description": "", "url": "", "extract": ""}


In [7]:
def build_wikipedia_graph(seed_article: str, max_links: int) -> nx.DiGraph:
    """
    Build a directed Wikipedia article graph at 1-hop depth.

    Nodes  : seed article + all articles it links to
    Edges  : seed->neighbor  and  neighbor->neighbor (cross-links)
    """
    G = nx.DiGraph()

    # ── Step 1: Seed node ──────────────────────────────────────────────
    print(f"\nStep 1: Fetching seed article metadata → '{seed_article}'")
    seed_meta = get_article_summary(seed_article)
    G.add_node(
        seed_article,
        description=seed_meta["description"],
        url=seed_meta["url"],
        extract=seed_meta["extract"],
        is_seed=True,
    )

    # ── Step 2: Outgoing links from seed ──────────────────────────────
    print(f"\nStep 2: Fetching outgoing links (max {max_links})...")
    neighbors = get_outgoing_links(seed_article, max_links)
    print(f"  → Found {len(neighbors)} linked articles")

    # ── Step 3: Add neighbor nodes + seed->neighbor edges ─────────────
    print(f"\nStep 3: Adding neighbor nodes and seed edges...")
    neighbor_set = set(neighbors)

    for title in neighbors:
        time.sleep(REQUEST_DELAY)
        meta = get_article_summary(title)
        G.add_node(
            title,
            description=meta["description"],
            url=meta["url"],
            extract=meta["extract"],
            is_seed=False,
        )
        G.add_edge(seed_article, title, relationship="seed_to_neighbor")

    # ── Step 4: Cross-links among neighbors ───────────────────────────
    print(f"\nStep 4: Detecting cross-links among {len(neighbors)} neighbors...")
    cross_link_count = 0

    for i, title in enumerate(neighbors):
        print(f"  [{i+1}/{len(neighbors)}] Checking cross-links for '{title}'")
        time.sleep(REQUEST_DELAY)

        try:
            neighbor_links = get_outgoing_links(title, max_links=500)
            for linked_title in neighbor_links:
                if linked_title in neighbor_set and linked_title != title:
                    if not G.has_edge(title, linked_title):
                        G.add_edge(title, linked_title, relationship="cross_link")
                        cross_link_count += 1
        except Exception as e:
            print(f"    ✗ Could not fetch links for '{title}': {e}")

    print(f"  → Found {cross_link_count} cross-links among neighbors")
    return G



In [8]:

def summarize_graph(G: nx.DiGraph) -> None:
    """Print a structured summary of the directed graph."""
    print("\n" + "=" * 60)
    print("GRAPH SUMMARY")
    print("=" * 60)
    print(f"  Nodes (articles)   : {G.number_of_nodes()}")
    print(f"  Edges (links)      : {G.number_of_edges()}")
    print(f"  Graph density      : {nx.density(G):.4f}")
    print(f"  Is weakly connected: {nx.is_weakly_connected(G)}")

    seed_edges  = [(u,v) for u,v,d in G.edges(data=True)
                   if d["relationship"] == "seed_to_neighbor"]
    cross_edges = [(u,v) for u,v,d in G.edges(data=True)
                   if d["relationship"] == "cross_link"]
    print(f"  Seed→neighbor edges: {len(seed_edges)}")
    print(f"  Cross-link edges   : {len(cross_edges)}")

    # Top by in-degree
    in_degrees = sorted(G.in_degree(), key=lambda x: -x[1])
    print(f"\nTop 10 by IN-DEGREE (most referenced internally):")
    print(f"  {'Article':<45} {'In-deg':>6}")
    print(f"  {'-'*45} {'-'*6}")
    for title, deg in in_degrees[:10]:
        print(f"  {title:<45} {deg:>6}")

    # Top by out-degree
    out_degrees = sorted(G.out_degree(), key=lambda x: -x[1])
    print(f"\nTop 10 by OUT-DEGREE (most links outward):")
    print(f"  {'Article':<45} {'Out-deg':>7}")
    print(f"  {'-'*45} {'-'*7}")
    for title, deg in out_degrees[:10]:
        print(f"  {title:<45} {deg:>7}")


In [9]:
def export_graph(G: nx.DiGraph, output_dir: str = "output") -> None:
    """Export graph to GraphML, edge CSV, and node CSV."""
    os.makedirs(output_dir, exist_ok=True)

    nx.write_graphml(G, os.path.join(output_dir, "wikipedia_graph.graphml"))

    edges = [
        {"source": u, "target": v, "relationship": d["relationship"]}
        for u, v, d in G.edges(data=True)
    ]
    pd.DataFrame(edges).to_csv(
        os.path.join(output_dir, "wikipedia_edges.csv"), index=False
    )

    nodes = [
        {
            "article":     n,
            "is_seed":     d.get("is_seed", False),
            "description": d.get("description", ""),
            "url":         d.get("url", ""),
            "in_degree":   G.in_degree(n),
            "out_degree":  G.out_degree(n),
        }
        for n, d in G.nodes(data=True)
    ]
    pd.DataFrame(nodes).to_csv(
        os.path.join(output_dir, "wikipedia_nodes.csv"), index=False
    )

    print(f"\nFiles saved to /{output_dir}/")
    print(f"  → wikipedia_graph.graphml")
    print(f"  → wikipedia_edges.csv")
    print(f"  → wikipedia_nodes.csv")


In [10]:
def main():
    print("WIKIPEDIA ARTICLE LINK GRAPH BUILDER")
    print(f"Seed article : '{SEED_ARTICLE}'")
    print(f"Max links    : {MAX_LINKS}")
    print(f"Depth        : 1 hop\n")

    G = build_wikipedia_graph(SEED_ARTICLE, MAX_LINKS)
    summarize_graph(G)
    export_graph(G)
    return G


if __name__ == "__main__":
    G = main()

WIKIPEDIA ARTICLE LINK GRAPH BUILDER
Seed article : 'Arsenal F.C.'
Max links    : 80
Depth        : 1 hop


Step 1: Fetching seed article metadata → 'Arsenal F.C.'

Step 2: Fetching outgoing links (max 80)...
  Fetching links from 'Arsenal F.C.'...
  → Found 80 linked articles

Step 3: Adding neighbor nodes and seed edges...

Step 4: Detecting cross-links among 80 neighbors...
  [1/80] Checking cross-links for '1. FC Magdeburg'
  Fetching links from '1. FC Magdeburg'...
  [2/80] Checking cross-links for '1886–87 Royal Arsenal F.C. season'
  Fetching links from '1886–87 Royal Arsenal F.C. season'...
  [3/80] Checking cross-links for '1887–88 Royal Arsenal F.C. season'
  Fetching links from '1887–88 Royal Arsenal F.C. season'...
  [4/80] Checking cross-links for '1888–89 Royal Arsenal F.C. season'
  Fetching links from '1888–89 Royal Arsenal F.C. season'...
  [5/80] Checking cross-links for '1888–89 in English football'
  Fetching links from '1888–89 in English football'...
  [6/80] Chec